# Ch12. Advanced Methods
**Forecasting: Principles & Practice (Python Edition)**  
Lab Notebook · [github.com/bcseong2/fpppy-labs](https://github.com/bcseong2/fpppy-labs)

In [1]:
%pip install statsforecast neuralforecast hierarchicalforecast mlforecast utilsforecast


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## [Slide 3] Case Study: Bank Call Volume

In [2]:
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import MSTL

model = MSTL(season_length=[169, 169 * 5])
sf = StatsForecast(models=[model], freq="5min")
sf = sf.fit(
    bank_calls.assign(y=lambda df: np.sqrt(df["y"]))
)
dcmp = sf.fitted_[0, 0].model_

/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'bank_calls' is not defined

## [Slide 5] MSTL: Forecasting Strategy

In [3]:
import pandas as pd

# Generate only business-hour timestamps (Mon-Fri, 7am-9pm)
start = bank_calls["ds"].max() + pd.offsets.Minute(5)
all_5min = pd.date_range(
    start, start + pd.offsets.Day(7), freq="5min"
)
fc_ds = all_5min.to_series().loc[lambda x: (
    x.dt.weekday.between(0, 4) &
    x.dt.strftime("%H:%M").between("07:00", "21:00")
)]

fc = sf.predict(h=len(fc_ds), level=[80, 95])
# Back-transform from sqrt scale
res = fc.select_dtypes(include=np.number)
fc = fc.assign(**res.transform(lambda x: x**2))

NameError: name 'bank_calls' is not defined

## [Slide 8] Dynamic Harmonic Regression: Code

In [4]:
from utilsforecast.feature_engineering import fourier, pipeline
from statsforecast.models import AutoARIMA
from functools import partial

features = [
    partial(fourier, season_length=169, k=10),
    partial(fourier, season_length=5 * 169, k=5),
]
bank_fourier, bank_futr = pipeline(
    bank_calls, features=features,
    freq="5min", h=len(fc_ds)
)

# Drop redundant Fourier features (sin5_845, cos5_845
# are identical to sin5_169*5 = exact collinear)
dropcols = ["sin5_845", "cos5_845"]
bank_fourier = bank_fourier.drop(columns=dropcols)
bank_futr    = bank_futr.drop(columns=dropcols)

model = AutoARIMA(d=0, seasonal=False, nmodels=5)
sf = StatsForecast(models=[model], freq="5min")
sf = sf.fit(bank_fourier)
fc_fourier = sf.predict(
    h=len(fc_ds), X_df=bank_futr, level=[80, 95]
)

NameError: name 'bank_calls' is not defined

## [Slide 10] Electricity Demand: Multiple Seasonalities

In [5]:
from statsforecast.models import AutoARIMA

features = [
    partial(fourier, season_length=2 * 24,         k=10),
    partial(fourier, season_length=2 * 24 * 7,     k=5),
    partial(fourier, season_length=2 * 24 * 7*365, k=3),
]
vic_fourier, vic_futr = pipeline(
    vic_elec_df, features=features, freq="30min",
    h=2 * 48
)

model = AutoARIMA(max_d=0, seasonal=False, nmodels=20)
sf = StatsForecast(models=[model], freq="30min")
sf = sf.fit(vic_fourier, target_col="Demand")

NameError: name 'vic_elec_df' is not defined

## [Slide 14] Prophet: Cement Production Example

In [6]:
from prophet import Prophet

m = Prophet(weekly_seasonality=False)
m.add_seasonality(name="quarterly", period=4, fourier_order=2)
m.fit(train)

future = m.make_future_dataframe(periods=10, freq="QS")
fc_prophet = (
    m.predict(future)
    [["ds", "yhat", "yhat_lower", "yhat_upper"]]
    .rename(columns={
        "yhat":       "Prophet",
        "yhat_lower": "Prophet-lo-80",
        "yhat_upper": "Prophet-hi-80",
    })
)

Importing plotly failed. Interactive plots will not work.


NameError: name 'train' is not defined

## [Slide 18] AutoARIMAProphet and NeuralProphet

In [7]:
from statsforecast.models import AutoARIMAProphet

m = AutoARIMAProphet()
m.add_seasonality(
    name="hourly",    period=48, fourier_order=5)
m.add_seasonality(
    name="subhourly", period=48*7, fourier_order=5)
m.fit(train)

future = m.make_future_dataframe(periods=48*2, freq="30T")
fc_ap = m.predict(future)

ImportError: cannot import name 'AutoARIMAProphet' from 'statsforecast.models' (/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/statsforecast/models.py)

## [Slide 23] VAR: US Consumption and Income

In [8]:
from statsmodels.tsa.api import VAR
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf

us_change = (
    pd.read_csv("data/US_change.csv", parse_dates=["ds"])
    .rename(columns={"y": "Consumption"})
    .drop(columns=["unique_id"])
    .set_index("ds").asfreq("QS")
)

model = VAR(us_change)
fit_aic = model.fit(ic="aic")
fit_bic = model.fit(ic="bic")
print("Lags selected by AIC:", fit_aic.k_ar)  # VAR(5)
print("Lags selected by BIC:", fit_bic.k_ar)  # VAR(1)

FileNotFoundError: [Errno 2] No such file or directory: 'data/US_change.csv'

## [Slide 25] VAR: Forecasting

In [9]:
lag_order = fit_aic.k_ar
y = us_change.tail(lag_order).to_numpy()

fc80 = fit_aic.forecast_interval(y, steps=8, alpha=0.2)
fc95 = fit_aic.forecast_interval(y, steps=8, alpha=0.05)

fc_var5 = pd.concat([
    pd.DataFrame({
        "unique_id": column,
        "ds": pd.date_range("2019-07-01", periods=8, freq="QS"),
        "VAR_5":      fc80[0][:, i],
        "VAR_5-lo-80": fc80[1][:, i],
        "VAR_5-hi-80": fc80[2][:, i],
    })
    for i, column in enumerate(["Consumption", "Income"])
])

NameError: name 'fit_aic' is not defined

## [Slide 27] VAR: Additional Applications

In [10]:
# Granger causality test
from statsmodels.tsa.stattools import grangercausalitytests
results = grangercausalitytests(
    us_change[["Consumption", "Income"]],
    maxlag=5
)

# Impulse response function
irf = fit_aic.irf(periods=10)
irf.plot(orth=False)

NameError: name 'us_change' is not defined

## [Slide 30] STL Decomposition for Bootstrapping

In [11]:
from statsforecast.models import MSTL

sf = StatsForecast(
    models=[MSTL(season_length=4)], freq="QS"
)
sf = sf.fit(cement)
dcmp = sf.fitted_[0, 0].model_.assign(
    ds=cement["ds"].to_numpy()
)

NameError: name 'cement' is not defined

In [12]:
def blocked_bootstrap(dcmp, block_size, num_sims):
    num_blocks = int(np.ceil(len(dcmp) / block_size))
    sim = {}
    for i in range(num_sims):
        blocks = []
        for _ in range(num_blocks):
            start = np.random.randint(
                0, len(dcmp) - block_size + 1)
            blocks.append(
                dcmp["remainder"].iloc[start:start+block_size]
            )
        sim[str(i)] = (
            dcmp["trend"] + dcmp["seasonal"]
            + pd.concat(blocks, ignore_index=True)[:len(dcmp)]
        )
    return pd.DataFrame(sim)

## [Slide 32] Bagged Forecasts

In [13]:
from functools import reduce

sim_df = blocked_bootstrap(dcmp, block_size=8, num_sims=100)
sim_df = sim_df.assign(
    unique_id="Cement", ds=cement["ds"].to_numpy()
)

sf = StatsForecast(
    models=[AutoETS(season_length=4)], freq="QS"
)
forecasts = []
for col in sim_df.drop(columns=["ds","unique_id"]).columns:
    train = (sim_df[["unique_id","ds",col]]
             .rename(columns={col: "y"}))
    fc = sf.forecast(df=train, h=10)
    forecasts.append(fc)

NameError: name 'dcmp' is not defined

## [Slide 34] Bagged Forecasts: Averaging

In [14]:
forecasts_df = reduce(
    lambda left, right: left.merge(right),
    forecasts
)

# Average across all bootstrapped forecasts
fc_bagged = (
    forecasts_df
    .set_index(["unique_id", "ds"])
    .mean(axis="columns")
    .rename("BaggedETS")
    .reset_index()
)

# Compare with direct ETS forecast
fc_ets = sf.forecast(df=cement, h=10)

NameError: name 'forecasts' is not defined

## [Slide 37] Summary: Modules Used in Ch12

In [15]:
# Multiple seasonality
from statsforecast.models import MSTL, AutoARIMA, AutoETS
from utilsforecast.feature_engineering import fourier, pipeline

# Prophet
from prophet import Prophet
from statsforecast.models import AutoARIMAProphet

# VAR
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.graphics.tsaplots import plot_acf

# Bootstrapping & bagging
import numpy as np
import pandas as pd
from functools import reduce

ImportError: cannot import name 'AutoARIMAProphet' from 'statsforecast.models' (/Data2/data/jc25/forecast/fpppy-labs/.venv/lib/python3.11/site-packages/statsforecast/models.py)